# Data Exploration and Augmentation Visualization

This notebook explores the ETIS-LaribPolypDB dataset and visualizes augmentation effects.

## Setup


In [ ]:

import sys
sys.path.append('../src')

import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import albumentations as A
from PIL import Image

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries loaded successfully")



## Dataset Statistics


In [ ]:
def analyze_dataset(img_dir, ann_dir):
    """
    Analyze dataset statistics.
    """
    images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    # Count images with annotations
    annotated_count = 0
    total_boxes = 0
    box_sizes = []
    
    for img_name in images:
        ann_name = img_name.replace('.jpg', '.xml').replace('.png', '.xml')
        ann_path = os.path.join(ann_dir, ann_name)
        
        if os.path.exists(ann_path):
            annotated_count += 1
            import xml.etree.ElementTree as ET
            tree = ET.parse(ann_path)
            root = tree.getroot()
            
            for obj in root.findall('object'):
                name = obj.find('name').text
                if name.lower() == 'polyp':
                    total_boxes += 1
                    bbox = obj.find('bndbox')
                    xmin = float(bbox.find('xmin').text)
                    ymin = float(bbox.find('ymin').text)
                    xmax = float(bbox.find('xmax').text)
                    ymax = float(bbox.find('ymax').text)
                    width = xmax - xmin
                    height = ymax - ymin
                    box_sizes.append((width, height))
    
    stats = {
        'total_images': len(images),
        'annotated_images': annotated_count,
        'total_polyps': total_boxes,
        'avg_polyps_per_image': total_boxes / annotated_count if annotated_count > 0 else 0,
        'box_sizes': box_sizes
    }
    
    return stats

# Analyze dataset (update paths as needed)
img_dir = '../data/ETIS-LaribPolypDB/images'
ann_dir = '../data/ETIS-LaribPolypDB/annotations'

if os.path.exists(img_dir):
    stats = analyze_dataset(img_dir, ann_dir)
    print("Dataset Analysis")
    print("="*50)
    print(f"Total images: {stats['total_images']}")
    print(f"Images with polyps: {stats['annotated_images']}")
    print(f"Total polyp annotations: {stats['total_polyps']}")
    print(f"Average polyps per image: {stats['avg_polyps_per_image']:.2f}")
else:
    print(f"Dataset not found at {img_dir}")
    print("Please download ETIS-LaribPolypDB dataset first")




## Visualize Polyp Size Distribution


In [ ]:

if 'stats' in locals() and len(stats['box_sizes']) > 0:
    widths = [s[0] for s in stats['box_sizes']]
    heights = [s[1] for s in stats['box_sizes']]
    areas = [w * h for w, h in stats['box_sizes']]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Width distribution
    axes[0].hist(widths, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Width (pixels)', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Polyp Width Distribution', fontsize=14)
    axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean: {np.mean(widths):.0f}px')
    axes[0].legend()
    
    # Height distribution
    axes[1].hist(heights, bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Height (pixels)', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Polyp Height Distribution', fontsize=14)
    axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.0f}px')
    axes[1].legend()
    
    # Area distribution
    axes[2].hist(areas, bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
    axes[2].set_xlabel('Area (pixels²)', fontsize=12)
    axes[2].set_ylabel('Frequency', fontsize=12)
    axes[2].set_title('Polyp Area Distribution', fontsize=14)
    axes[2].axvline(np.mean(areas), color='red', linestyle='--', label=f'Mean: {np.mean(areas):.0f}px²')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig('../reports/polyp_size_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nPolyp Size Statistics:")
    print(f"  Width: mean={np.mean(widths):.0f}px, std={np.std(widths):.0f}px")
    print(f"  Height: mean={np.mean(heights):.0f}px, std={np.std(heights):.0f}px")
    print(f"  Area: mean={np.mean(areas):.0f}px², std={np.std(areas):.0f}px²")



## Augmentation Visualization


In [ ]:

def visualize_augmentations(image_path, num_variants=8):
    """
    Apply and visualize different augmentations.
    """
    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Define augmentations
    augmentations = [
        ("Original", None),
        ("RandomRotate90", A.RandomRotate90(p=1.0)),
        ("HorizontalFlip", A.HorizontalFlip(p=1.0)),
        ("VerticalFlip", A.VerticalFlip(p=1.0)),
        ("HueSaturationValue", A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=30, p=1.0)),
        ("RandomBrightnessContrast", A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0)),
        ("CLAHE", A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0)),
        ("GaussianBlur", A.GaussianBlur(blur_limit=3, p=1.0))
    ]
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx, (name, aug) in enumerate(augmentations[:num_variants]):
        if aug is None:
            augmented = image
        else:
            augmented = aug(image=image)['image']
        
        axes[idx].imshow(augmented)
        axes[idx].set_title(name, fontsize=10)
        axes[idx].axis('off')
    
    plt.suptitle('Data Augmentation Techniques for Polyp Detection', fontsize=16)
    plt.tight_layout()
    plt.savefig('../reports/augmentation_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

# Find a sample image
sample_images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
if len(sample_images) > 0:
    sample_path = os.path.join(img_dir, sample_images[0])
    visualize_augmentations(sample_path)



## HSV Color Space Augmentation


In [ ]:
def visualize_hsv_augmentation(image_path, num_variants=6):
    """
    Visualize HSV space augmentations specifically.
    """
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    variations = [
        ("Original", (0, 1, 1)),
        ("Hue +30", (30, 1, 1)),
        ("Hue -30", (-30, 1, 1)),
        ("Saturation 0.5", (0, 0.5, 1)),
        ("Saturation 1.5", (0, 1.5, 1)),
        ("Value 0.6", (0, 1, 0.6))
    ]
    
    for idx, (title, (h_shift, s_scale, v_scale)) in enumerate(variations):
        hsv_copy = image_hsv.copy().astype(np.float32)
        hsv_copy[:, :, 0] = (hsv_copy[:, :, 0] + h_shift) % 180
        hsv_copy[:, :, 1] = np.clip(hsv_copy[:, :, 1] * s_scale, 0, 255)
        hsv_copy[:, :, 2] = np.clip(hsv_copy[:, :, 2] * v_scale, 0, 255)
        hsv_copy = hsv_copy.astype(np.uint8)
        
        augmented = cv2.cvtColor(hsv_copy, cv2.COLOR_HSV2RGB)
        
        axes[idx].imshow(augmented)
        axes[idx].set_title(title, fontsize=11)
        axes[idx].axis('off')
    
    plt.suptitle('HSV Color Space Augmentations', fontsize=16)
    plt.tight_layout()
    plt.savefig('../reports/hsv_augmentation.png', dpi=150, bbox_inches='tight')
    plt.show()

if len(sample_images) > 0:
    visualize_hsv_augmentation(sample_path)



## Sample Images with Bounding Boxes


In [ ]:
def visualize_annotations(img_dir, ann_dir, num_samples=6):
    """
    Display sample images with their bounding boxes.
    """
    images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
    selected = np.random.choice(images, min(num_samples, len(images)), replace=False)
    
    cols = 3
    rows = (len(selected) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4*rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols > 1 else [axes]
    
    for idx, img_name in enumerate(selected):
        img_path = os.path.join(img_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Load annotation
        ann_name = img_name.replace('.jpg', '.xml').replace('.png', '.xml')
        ann_path = os.path.join(ann_dir, ann_name)
        
        if os.path.exists(ann_path):
            import xml.etree.ElementTree as ET
            tree = ET.parse(ann_path)
            root = tree.getroot()
            
            for obj in root.findall('object'):
                name = obj.find('name').text
                if name.lower() == 'polyp':
                    bbox = obj.find('bndbox')
                    xmin = int(float(bbox.find('xmin').text))
                    ymin = int(float(bbox.find('ymin').text))
                    xmax = int(float(bbox.find('xmax').text))
                    ymax = int(float(bbox.find('ymax').text))
                    cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
                    cv2.putText(image, 'Polyp', (xmin, ymin-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        axes[idx].imshow(image)
        axes[idx].set_title(f'{img_name}', fontsize=10)
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(len(selected), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Sample Images with Polyp Annotations', fontsize=16)
    plt.tight_layout()
    plt.savefig('../reports/sample_annotations.png', dpi=150, bbox_inches='tight')
    plt.show()

if os.path.exists(ann_dir):
    visualize_annotations(img_dir, ann_dir)



## Summary Statistics Output


In [ ]:

print("\n" + "="*60)
print("DATA EXPLORATION SUMMARY")
print("="*60)
print(f"""
Dataset: ETIS-LaribPolypDB
Source: Originally published by ETIS laboratory

Key Findings:
- Polyp sizes vary significantly (small to medium)
- HSV augmentations help with endoscopic lighting variations
- Geometric transforms address different polyp orientations

Recommended Augmentation Pipeline:
1. Random rotation (90°, 180°, 270°)
2. Horizontal/Vertical flip
3. HSV adjustments (hue ±20, saturation ±30%)
4. Brightness/Contrast (±20%)
5. Resize to 300×300 or 512×512
6. Normalization with ImageNet stats

Note: ETIS dataset is challenging due to:
- Small polyp sizes in some images
- Specular reflections
- Complex mucosal patterns
""")
print("="*60)
